# Flow Matching 配合 DiT 的实现与改进

我们已经为 Diffusion 铺设了完整的基础框架，本章我们来看一些极其成功的实现案例以及 Flow Matching 的改进。

# Stable Diffusion 3

推荐你读 https://arxiv.org/abs/2403.03206 Scaling Rectified Flow Transformers for High-Resolution Image Synthesis 这是 SD3 的技术报告。简而言之，SD3 是我们说的 FM 结合 DiT 的集大成者。

SD3 的最大改进在于指出 FM 训练时均匀采样时间步 $t$ 所带来的问题，以及改进了 Cross-Attention 模块。我们一一来说。

训练 FM 模型时，我们一般采取高斯路径中的最优传输路径 $x_t = (1-t)z + t\epsilon$。在第五章我们提到，原损失函数 $\mathcal{L}_{FM}$ 等价条件路径损失函数 $\mathcal{L}_{CFM}$。其中

$$L_{\text{CFM}}(\theta) = \mathbb{E}_{t \sim \text{Unif}, z \sim p_{\text{data}}, x \sim p_t(\cdot | z)} \left[ \| u_t^{\theta}(x) - u_t^{\text{target}}(x | z) \|^2 \right]$$ $$= \mathbb{E}_{t \sim \text{Unif}, z \sim p_{\text{data}}, \epsilon \sim \mathcal{N}(0, I_d)} \left[ \| u_t^{\theta}(\mu_t z + \sigma_t \epsilon) - u_t^{\text{target}}(\mu_t z + \sigma_t \epsilon | z) \|^2 \right]$$ $$= \mathbb{E}_{t \sim \text{Unif}, z \sim p_{\text{data}}, \epsilon \sim \mathcal{N}(0, I_d)} \left[ \| u_t^{\theta}(\mu_t z + \sigma_t \epsilon) - (\dot{\mu}_t z + \dot{\sigma}_t \epsilon) \|^2 \right]$$
此处的 $\mu_t = 1 - t$，$\sigma_t = t$。符号上可能与第五章有所出入，但是他们是等价的。

注意看对条件路径损失的定义。我们要求时间步 $t$ 在 $[0,1]$ 内均匀采样计算期望。但是这里实际上存在问题，在 $t \to 0$ 或 $t \to 1$ 时，条件路径指向的目标非常明确，神经网络很容易学习到矢量场。然而，在 $t \approx 0.5$ 附近，图像正处于从噪声中逐渐显影的时刻。此时预测目标 $\epsilon - x_0$ 包含复杂的语义信息，模型在此处学习矢量场比较困难。因此我们可以改进训练时对于时间步 $t$ 的采样方式，将均匀采样改为采样中间概率更大的方案。

我们记采样密度函数为 $\pi$。SD3 原论文给出几种方案，我们简单介绍。

### Logit-Normal Sampling

从标准正态分布采样 $u \sim \mathcal{N}(m, s^2)$，通过 $\text{sigmoid}$ 函数将其映射到 $[0, 1]$，即 $t = \sigma(u)$。最终密度函数是$$\pi_{ln}(t; m, s) = \frac{1}{s\sqrt{2\pi}} \frac{1}{t(1-t)} \exp\left(-\frac{(\text{logit}(t)-m)^2}{2s^2}\right)$$
其中 $\text{logit}(t) = \log \frac{t}{1-t}$。

这个函数看起来像一个平滑的钟型。当 $m=0$ 时，分布关于 $t=0.5$ 完美对称。

### Mode Sampling with Heavy Tails

Logit-Normal 的一个缺陷是它在 $t=0$ 和 $t=1$ 处的密度会迅速衰减到 0，这意味着模型在训练时几乎看不见纯图像和纯噪声。作者做出改进。通过一个单调函数 $f_{mode}(u; s)$ 将均匀分布 $u \in [0, 1]$ 映射到 $t$ $$f_{mode}(u; s) = 1 - u - s \cdot (\cos^2(\frac{\pi}{2}u) - 1 + u)$$
密度函数形式与 Logit-Normal 一致。

### CosMap

直接给出密度函数 $$\pi_{CosMap}(t) = \frac{2}{\pi - 2\pi t + 2\pi t^2}$$
这是一个相对扁平的凹形或微凸曲线。

我们基本说完了。总之就是想方设法改进采样时间步的密度函数。这里其实非常工程化。

下面我们来说说改进的 Cross-Attention 模块，正式名称是 Multimodal Diffusion Transformer。

## Multimodal Diffusion Transformer

下面这张图详细展示了 SD3 的整体架构与具体的 MM-DiT Block 内部架构。

<img src="./assets/MM-DiT.png" width="800" height="550">

说实话略复杂，我为你梳理清楚。先看整体架构。最右侧是初始高斯噪声的 Patchify 与位置编码输入。最左侧则是通过正余弦位置编码与 MLP 编码时间步。最上面则是对文本标签的编码。我们详细说说对文本的编码。

编码文本非常奢侈地同时使用了三个编码器 CLIP-L/14, CLIP-G/14 和 T5-XXL。其中，CLIP 编码器是基于图像-文本对齐训练的，提供视觉对齐支持，T5 作为纯语言编码器则提供细致的语义理解。CLIP 的编码不仅仅会进入标签编码 $c$，还会与时间步编码混合作为编码 $y$，理由是与时间步的编码可以作为宏观信息，而单独编码则可以更详细地指导。

需要注意，$77 + 77$ 这个注解实际上因为 CLIP 原始设计只能编码 $77$ 个 token，因此我们使用两个 CLIP 编码器拼接编码。关于每个编码器的详细编码能力，实际上 CLIP-L 的向量长度为 768，CLIP-G 的向量长度为 1280，T5-XXL 的向量长度高达 4096。原因是 T5-XXL 是一个 11B 大小的编码器，可以为语义提供无比详细的信息。

关于这三个编码器编码维度不一致，我们会采用一个线性层转换到 MM-DiT 的工作维度再在序列长度维度上拼接，比如 1536 或 2048。

我们现在来说说 MM-DiT Block。

首先请观察左右两侧的对于标签编码 $y$ 的处理，这里实际上就是上一章提到的 adaLN-Zero。我们将 $y$ 编码为 $6$ 个参数向量，通过 Scale 与 Shift 调制张量，我们不多描述。

关于中间的这部分，其被称为 Adjoint Attention。在这里，文本编码 $c$ 与图像编码 $x$ 拥有完全相同对称的处理地位与方式。在最终的 Attention 模块，文本编码与图像编码先得到各自的注意力矩阵 $Q$, $K$ 与 $V$，然后直接将相对应的矩阵按照序列维度进行拼接，得到新的注意力矩阵。换句话说，文本的 $Q, K, V$ 和图像的 $Q, K, V$ 会被按照序列维度拼接在一起，在随后的注意力计算中，图像编码会关注所有文本编码，文本编码也会关注所有图像编码。这种双向信息流是 SD3 语义遵循能力强大的根本原因。

在注意力计算完结之后，我们重新将拼接的超长序列张量分割回原来的两个张量，再各自分流处理。

关于 SD3 的核心技术，我们基本说完了。

接下来我想说说 Rectified Flow。实际上 SD3 使用的训练算法并不是原始的 Flow Matching，而是一种改进。

# Rectified Flow

推荐你读 https://arxiv.org/abs/2209.03003 Flow Straight and Fast: Learning to Generate and Transfer Data with Rectified Flow 这是 Reflow 原论文。

诡异的是，不必将 Reflow 视为 Flow Matching 的补丁，因为 Reflow 的发布时间甚至比 Flow Matching 更早。他们之间的工作更像雷同而不是补充。Reflow 的主要价值是提出了一种后训练方法，通过将训练数据配对，可以将推理轨迹拉直，从而减少推理步数加速推理。

但是为什么现在普遍将 Reflow 的工作视为一种必要的后训练方法？或者换句话说，为什么我们对 FM 仍有不满之处？

首先我们指出一件事，那就是 FM 训练得到的轨迹还不够直。我们在训练时已经尽力让模型学习最优传输路径，然而，由于我们训练时随机地采样初始噪声并且将其与数据集中数据点配对训练，这导致模型学习到的路径极其容易交错。模型为了保证损失最小，交错的轨迹则会导致学习到的矢量场的指向模糊。下面这张图来自 Reflow 原论文，指出这种交错带来的问题。

<img src="./assets/Reflow.png" width="1000" height="230">

上面四幅图中，最左侧是我们预设的最优传输路径，这些路径天生地交错在一起。第二张图是经过 FM 训练之后结果，由于随机配对的训练，模型实际学习到的路径发生了弯曲。因此，我们决定进行 Reflow 后训练。我们将原本上下各自噪声出发的真实图片配对再次训练，模型就发现了最快速的直线路径，这与我们所想要的完全一致。

### 训练与目标函数

我们首先定义一些符号。

定义重训练算子 $\text{RectFlow}(\cdot, \cdot)$，作为映射接收训练起始数据点与训练终点数据点，表示训练过程所用的数据，输出则是流模型。

定义 $\mathbf{Z}^k$ 用来指代第 $k$ 代流模型，指的就是经过 $k$ 次重训练的流模型。而 $\mathbf{Z}^1$ 就是第一代流模型，也就是原 FM 模型。我们用 $\mathbf{Z}_0^k$ 指代 $\mathbf{Z}^k$ 使用的起始数据点，用 $\mathbf{Z}_1^k$ 来指代 $\mathbf{Z}^k$ 产生的终点数据点。

其中，在 $k \ge 1$ 时，$\mathbf{Z}_0^k$ 一般是被直接设定为遵从高斯分布的纯噪声，$\mathbf{Z}_1^k$ 则是流模型 $\mathbf{Z}^k$ 从起始数据点 $\mathbf{Z}_0^k$ 出发推理得到的结果。

当 $k$ 为 $0$，$(\mathbf{Z}_0^0, \mathbf{Z}_1^0)$ 就是原始的噪声分布采样得到的某点与训练数据集中某点 $(X_0, X_1)$。实际上对任意 $k$ 一般有 $X_0 = \mathbf{Z}_0^k$。我们认为样本遵从概率分布 $X_0 \sim \pi_0, X_1 \sim \pi_1$。其中 $\pi_0$ 是初始高斯分布，而 $\pi_1$ 是真实数据分布 $p_{data}$。

因此，存在关系 $\mathbf{Z}^{k+1} = \text{RectFlow}(\mathbf{Z}_0^k, \mathbf{Z}_1^k)$。

在讲 Reflow 提出的后训练方法之前，我想先说说 Reflow 如何做首次训练，就是从初始化的模型开始训练。后训练与首次训练实际上是共通的。

在 $t \in [0,1]$ 下，定义 ODE $$\mathrm{d}Z_t = v(Z_t, t)\mathrm{d}t$$
这是标准的概率流 ODE。定义线性插值点 $X_t = tX_1 + (1-t)X_0$。我们想要的是，模型在起点与终点之间的每一线性插值点 $(\mathbf{X}_t,t)$ 都给出指向终点的矢量方向。因此我们自然地给出损失函数 $$\min_{v} \int_0^1 \mathbb{E}_{ X_0 \sim \pi_0, X_1 \sim \pi_1}\left[ \|(X_1 - X_0) - v(X_t, t)\|^2 \right] \mathrm{d}t$$
或者如果我们将积分写入期望 $$\min_{v} \mathbb{E}_{t \sim \text{Uniform}[0, 1], X_0 \sim \pi_0, X_1 \sim \pi_1}\left[ \|(X_1 - X_0) - v(X_t, t)\|^2 \right] \mathrm{d}t$$

这很容易理解，因为这实际上就是要求任意时刻速度保持直线指向 $$\frac{dX_t}{dt} = \frac{d}{dt} \left[ X_0 + t(X_1 - X_0) \right] = X_1 - X_0$$

重要提醒，这个损失函数形式我们实际上极其熟悉。回顾一下第四章，在原始 Flow Matching 框架下，考虑最优传输路径 $x = \sigma_t x_0 + \mu_t(z)$，其中 $\mu_t = t$ 且 $\sigma_t = 1-t$。考虑我们推导的 FM 条件损失函数形式 $$L_{\text{CFM}}(\theta) = \mathbb{E}_{t \sim \text{Uniform}[0,1], z \sim p_{\text{data}}, \epsilon \sim \mathcal{N}(0, I_d)} \left[ \| (\dot{\mu}_t z + \dot{\sigma}_t \epsilon) - u_t^{\theta}(\mu_t z + \sigma_t \epsilon)\|^2 \right]$$
$$= \mathbb{E}_{t \sim \text{Uniform}[0,1], z \sim p_{\text{data}}, \epsilon \sim \mathcal{N}(0, I_d)} \left[ \|  (z - \epsilon) - u_t^{\theta}(x_t)\|^2 \right]$$

这与 Reflow 的损失函数完全一致。这证明一件事：在对于初始数据分布 $X_0 \sim \pi_0, X_1 \sim \pi_1$ 训练流模型时，Reflow 和 Flow Matching 完完全全一模一样。

所以 Reflow 真正的魅力在哪？答案是后训练，也就是对于 $k \ge 1$ 情况下的训练。

我们重申一下。在 $k \ge 1$ 时，$\mathbf{Z}_0^k$ 一般是被直接设定为遵从高斯分布的纯噪声，$\mathbf{Z}_1^k$ 则是流模型 $\mathbf{Z}^k$ 从这个噪声出发推理得到的数据集。换句话说，流模型认为噪声 $\mathbf{Z}^k$ 所在的路径更偏向于指向图像 $\mathbf{Z}_1^k$。而 Reflow 的目的就是加强这种偏爱的绑定关系直至变为直线。

再次定义线性插值点 $X^{new}_t = tZ_1 + (1-t)Z_0$。此时损失函数变为
$$\min_{v} \int_0^1 \mathbb{E}_{ Z_0 \sim \pi_0, Z_1 = \text{Inference}(Z_0)}\left[ \|(Z_1 - Z_0) - v(X^{new}_t, t)\|^2 \right] \mathrm{d}t$$

现在我们从头来讲述如何做后训练。假设我们拥有流模型 $\mathbf{Z}^k$。

首先采样 $n$ 个高斯噪声 $\mathbf{Z}^k_0 \sim \mathcal{N}(0, \mathbf{I})$，其中 $n$ 必须足够大以覆盖数据集，如在 CIFAR 上是 $50000$，在规模更庞大的模型上可能更大。

使用当前的 $\mathbf{Z}^k$ 模型，对每一个噪声 $\mathbf{Z}^k_0$ 进行完整的 ODE 求解。其中求解需要用一个比较高精度的求解器，如 Euler 求解器求解 $50$ 步，把这些噪声从 $t=0$ 推理到 $t=1$。我们得到了 $n$ 个对应的生成图像 $\mathbf{Z}_1^k$。

现在我们得到 $n$ 个配对训练数据对 $(\mathbf{Z}^k_0,\mathbf{Z}_1^k)$。对于每个对，随机采样一个时间步 $t \sim \mathcal{U}[0,1]$，计算线性插值结果 $\mathbf{X}^{new}_t$。将 $\mathbf{X}^{new}_t$ 与时间步 $t$ 输入网络得到预测的矢量场 $v(X^{new}_t, t)$。

最后根据损失公式计算损失，遍历一个 Batch 之后计算加权平均损失，反向传播更新参数。

值得一提，关于时间步 $t$ 的采样方式，Reflow 当然也可以遵循 SD3 做的诸多改进。

以下是完整的训练算法。

$$\begin{array}{l}
\hline
\textbf{Algorithm 1 } \text{Rectified Flow: Main Algorithm} \\
\hline
\textbf{Procedure: } \mathbf{Z} = \text{RectFlow}((X_0, X_1)): \\
\quad \textit{Inputs: } \text{Draws from a coupling } (X_0, X_1) \text{ of } \pi_0 \text{ and } \pi_1; \text{ velocity model } v_\theta: \mathbb{R}^d \to \mathbb{R}^d \text{ with parameter } \theta. \\
\quad \textit{Training: } \hat{\theta} = \arg\min_{\theta} \mathbb{E} \left[ \left\| X_1 - X_0 - v(tX_1 + (1 - t)X_0, t) \right\|^2 \right], \text{ with } t \sim \text{Uniform}([0, 1]). \\
\quad \textit{Sampling: } \text{Draw } (Z_0, Z_1) \text{ following } dZ_t = v_{\hat{\theta}}(Z_t, t)dt \text{ starting from } Z_0 \sim \pi_0 \text{ (or backwardly } Z_1 \sim \pi_1). \\
\quad \textit{Return: } \mathbf{Z} = \{Z_t : t \in [0, 1]\}. \\
\textbf{Reflow } \text{(optional): } \mathbf{Z}^{k+1} = \text{RectFlow}((Z_0^k, Z_1^k)), \text{ starting from } (Z_0^0, Z_1^0) = (X_0, X_1). \\
\textbf{Distill } \text{(optional): } \text{Learn a neural network } \hat{T} \text{ to distill the } k\text{-rectified flow, such that } Z_1^k \approx \hat{T}(Z_0^k). \\
\hline
\end{array}$$

### 原理

我们简单说说，为什么 Reflow 提出的后训练方法是有效的。由于我们已经说明其在首次训练时等价 Flow Matching，我们只关心后训练。

下面这张图展示了后训练拉直推理轨迹的过程。直觉上，这种拉直本质是在减少能量传输的损耗。

<img src="./assets/straight.png" width="1000" height="260">

我们现在开始着手证明后训练的有效性。定义中间状态 $$Z_t = Z_0 + \int_0^t v^{\mathbf{X}}(Z_t, t) \mathrm{d}t, \quad \forall t \in [0, 1], \quad Z_0 = X_0$$
一个结论是，$Z_t$ 与线性插值结果 $X^{new}_t$ 拥有完全相同的边缘分布。

关于这个结论，我们仅仅给出直觉性证明。由于 $Z_t$ 与 $X^{new}_t$ 遵守相同的 ODE 演变诱导的连续性方程，且初始均为 $Z_0$，他们的边缘分布是相同的。


定义轨迹 $\mathbf{Z} = \{Z_t\}$，定义测量直线程度函数 $$S(\mathbf{Z}) = \int_0^1 \mathbb{E} \left[ \|(Z_1 - Z_0) - \dot{Z}_t\|^2 \right] \mathrm{d}t$$
那么对于第 $k$ 代流模型的从 $Z_0 = X_0$ 出发推理产生的轨迹，有 $$\min_{k \in \{0,...,K\}} S(\mathbf{Z}^k) \le \frac{\mathbb{E}[\|X_1 - X_0\|^2]}{K} \quad (*)$$
这个结论是最为关键的，因为其证明了 Reflow 在拉直推理的轨迹。最小弯曲程度正在不断减小，这就是后训练有效的证据。

我们证明这个结论需要用到两个引理。定义耦合方差 $V((Z^k_0, Z^k_1))$ $$V = \int_0^1 \mathbb{E} \left[ \|(Z^k_1 - Z^k_0) - v^X(X^{new}_t, t)\|^2 \right] \mathrm{d}t$$

首先第一个需要用到的引理是 $${\int_0^1 \mathbb{E} [ \|\dot{Z}_t^{k+1}\|^2 ] \mathrm{d}t} = \mathbb{E} [\|Z_1^k - Z_0^k\|^2] - V((Z_0^k, Z_1^k))$$
这是因为 $$\mathbb{E} [\|Z_1^k - Z_0^k\|^2] $$ $$= \mathbb{E} [\|v^{k+1}(X^{new}_t, t)\|^2] + \mathbb{E} [\|(Z_1^k - Z_0^k) - v^{k+1}(X^{new}_t, t)\|^2] + 2 \mathbb{E} [\langle v^{k+1}(X^{new}_t, t), (Z_1^k - Z_0^k) - v^{k+1}(X^{new}_t, t) \rangle]$$
对于第三项交叉项有 $$\mathbb{E} [ (Z_1^k - Z_0^k) - v^{k+1}(X^{new}_t, t) ] = 0$$
因此交叉项为 $0$ $$\mathbb{E} [\|Z_1^k - Z_0^k\|^2] = \mathbb{E} [\|v^{k+1}(X^{new}_t, t)\|^2] + \mathbb{E} [\|(Z_1^k - Z_0^k) - v^{k+1}(X^{new}_t, t)\|^2]$$
现在积分 $$\int_0^1 \mathbb{E} [\|Z_1^k - Z_0^k\|^2] \mathrm{d}t = \int_0^1 \mathbb{E} [\|v^{k+1}(X^{new}_t, t)\|^2] \mathrm{d}t + \int_0^1 \mathbb{E} [\|(Z_1^k - Z_0^k) - v^{k+1}(X^{new}_t, t)\|^2] \mathrm{d}t$$
得到形式 $$\mathbb{E} [\|Z_1^k - Z_0^k\|^2] = \int_0^1 \mathbb{E} [\|v^{k+1}(X^{new}_t, t)\|^2] \mathrm{d}t + V((Z_0^k, Z_1^k))$$
此处用上我们刚刚的结论 $$\mathbb{E}_{X^{new}_t} [ \|v^{k+1}(X^{new}_t, t)\|^2 ] = \mathbb{E}_{Z_t^{k+1}} [ \|v^{k+1}(Z_t^{k+1}, t)\|^2 ] = \mathbb{E} [ \|\dot{Z}_t^{k+1}\|^2 ]$$
最终得证 $$\mathbb{E} [\|Z_1^k - Z_0^k\|^2] = \int_0^1 \mathbb{E} [ \|\dot{Z}_t^{k+1}\|^2 ] \mathrm{d}t + V((Z_0^k, Z_1^k))$$

现在我们指出第二个需要用到的引理。

对于原测量函数，有 $$\mathbb{E} [\|Z_1^k - Z_0^k\|^2] - \mathbb{E} [\|Z_1^{k+1} - Z_0^{k+1}\|^2] = S(\mathbf{Z}^{k+1}) + V((Z_0^k, Z_1^k))$$
这是因为 $$S(\mathbf{Z}^{k+1}) = \int_0^1 \mathbb{E} \left[ \|(Z_1^{k+1} - Z_0) - \dot{Z}_t^{k+1}\|^2 \right] \mathrm{d}t$$
展开得到 $$S(\mathbf{Z}^{k+1}) = {\int_0^1 \mathbb{E} [ \|Z_1^{k+1} - Z_0\|^2 ] \mathrm{d}t} + {\int_0^1 \mathbb{E} [ \|\dot{Z}_t^{k+1}\|^2 ] \mathrm{d}t} - {2 \int_0^1 \mathbb{E} [ \langle Z_1^{k+1} - Z_0, \dot{Z}_t^{k+1} \rangle ] \mathrm{d}t}$$
其中第一项就是 $$\mathbb{E} [ \|Z_1^{k+1} - Z_0\|^2 ]$$
第二项用到第一个结论 $${\int_0^1 \mathbb{E} [ \|\dot{Z}_t^{k+1}\|^2 ] \mathrm{d}t} = \mathbb{E} [\|Z_1^k - Z_0^k\|^2] - V((Z_0^k, Z_1^k))$$
第三项 $$2 \mathbb{E} [ \langle Z_1^{k+1} - Z_0, Z_1^{k+1} - Z_0 \rangle ] = 2 \mathbb{E} [ \|Z_1^{k+1} - Z_0\|^2 ]$$
因此成立。

现在我们用上这两个引理来证明最终的结论。

累计 $$\sum_{k=0}^K \left( \mathbb{E}[\|Z_1^k - Z_0^k\|^2] - \mathbb{E}[\|Z_1^{k+1} - Z_0^{k+1}\|^2] \right) = \sum_{k=0}^K \left( S(\mathbf{Z}^{k+1}) + V((Z_0^k, Z_1^k)) \right)$$
左边相消得到 $$\mathbb{E}[\|X_1 - X_0\|^2] - \mathbb{E}[\|Z_1^{K+1} - Z_0^{K+1}\|^2]$$
因此 $$\sum_{k=0}^K S(\mathbf{Z}^{k+1}) + V((Z_0^k, Z_1^k)) \le \mathbb{E}[\|X_1 - X_0\|^2]$$
随着 $K$ 增大，每一项 $S$ 和 $V$ 必须趋于 0。最终得到 $$\min_{k \in \{0,..., K\}} S(\mathbf{Z}^k) \le \frac{\mathbb{E}[\|X_1 - X_0\|^2]}{K}$$
结论得证。

如你所见，推导工作不是特别容易。

# 总结

Reflow 技术如今被广泛应用在加速生成领域，尽管构造数据集需要巨大的算力，但是带来的便利是一劳永逸的。在极小的步数内，模型可以生成极高质量的图像。

可能出乎意料，SD3 系列模型最大的参数量只有 8B，甚至比 T5-XXL 编码器更小。原因是画图模型的强项是处理空间, 色彩分布和构图。而理解人类复杂的语言逻辑需要极高的语义建模能力，借用现成的编码器比起完整训练会简单得多。

值得一提，SD3 在 2024 年由于其极小的参数体积与强大的性能引起了广泛关注，但是在完全开源之后被逐渐发现问题并且落伍。

本章内容可能很熟悉，轻车熟路。接下来我想要介绍一些目前的生成或推理优化技术。我想先从加速生成开始说起，Consistency Models。